### Import Libraries 

In [40]:
import pandas as pd
import numpy as np

# Visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# Model Data Preprocessing & Encoding
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, FunctionTransformer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Model Training
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Model Evaluation
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, roc_curve, classification_report,confusion_matrix

# Model Explainability 
import shap

### Load Data

In [24]:
df_model_data = pd.read_parquet(r"C:\Users\leest\OneDrive\Data Projects\Amdari Projects\Digital_Transfer_Fraud_Detection_System\Datasets\Artifats\Modelled_data")

In [25]:
df_model_data.head(3)

,timestamp,home_country,source_currency,dest_currency,channel,amount_usd,new_device,ip_country,location_mismatch,ip_risk_score,...,txn_day_of_week,late_night_hours,amount_high,critical_ip_risk,critical_device_trust,new_account,velocity_spike,chargeback_flag,low_kyc,web_high_amount
0,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,atm,278.19,False,US,False,0.123,...,Monday,0,0,0,0,0,0,0,0,0
1,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,154.29,True,CA,False,0.569,...,Monday,0,0,0,0,0,0,0,0,0
2,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,False,US,False,0.437,...,Monday,0,0,0,0,0,0,0,0,0


In [26]:
df_model_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10940 entries, 0 to 10939
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype              
---  ------                 --------------  -----              
 0   timestamp              10940 non-null  datetime64[ns, UTC]
 1   home_country           10940 non-null  object             
 2   source_currency        10940 non-null  object             
 3   dest_currency          10940 non-null  object             
 4   channel                10940 non-null  object             
 5   amount_usd             10940 non-null  float64            
 6   new_device             10940 non-null  bool               
 7   ip_country             10940 non-null  object             
 8   location_mismatch      10940 non-null  bool               
 9   ip_risk_score          10940 non-null  float64            
 10  kyc_tier               10940 non-null  object             
 11  account_age_days       10940 non-null  int64          

### Train-Test Split & Preprocessing

In [27]:
# Sort timestamp chronologically first
df_model_data = df_model_data.sort_values("timestamp").reset_index(drop=True)

# 80/20 split, in chronological order
split_index = int(len(df_model_data) * 0.8)

train_df = df_model_data.iloc[:split_index]
test_df = df_model_data.iloc[split_index:]

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nTrain period: {train_df['timestamp'].min()} to {train_df['timestamp'].max()}")
print(f"Test period:  {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")
print(f"\nTrain fraud rate: {train_df['is_fraud'].mean()*100:.2f}%")
print(f"Test fraud rate:  {test_df['is_fraud'].mean()*100:.2f}%")

Train shape: (8752, 28)
Test shape:  (2188, 28)

Train period: 2022-10-03 18:40:59.468549+00:00 to 2025-03-23 15:23:26.468549+00:00
Test period:  2025-03-23 20:11:35.468549+00:00 to 2025-12-16 00:13:41.468549+00:00

Train fraud rate: 7.77%
Test fraud rate:  14.12%


In [28]:
# Drop target feature

df_model_data = df_model_data.drop(["is_fraud"], axis=1)

In [34]:
categorical_features = df_model_data.select_dtypes(include=["object", "bool"]).columns
categorical_features

Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'new_device', 'ip_country', 'location_mismatch', 'kyc_tier',
       'txn_day_of_week'],
      dtype='object')

In [35]:
numerical_features = df_model_data.select_dtypes(include=["float", "int"]).columns
numerical_features

Index(['amount_usd', 'ip_risk_score', 'account_age_days', 'device_trust_score',
       'risk_score_internal', 'txn_velocity_24h', 'corridor_risk', 'txn_hour',
       'late_night_hours', 'amount_high', 'critical_ip_risk',
       'critical_device_trust', 'new_account', 'velocity_spike',
       'chargeback_flag', 'low_kyc', 'web_high_amount'],
      dtype='object')

In [36]:
all_features = list(categorical_features) + list(numerical_features)
all_features

['home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'new_device',
 'ip_country',
 'location_mismatch',
 'kyc_tier',
 'txn_day_of_week',
 'amount_usd',
 'ip_risk_score',
 'account_age_days',
 'device_trust_score',
 'risk_score_internal',
 'txn_velocity_24h',
 'corridor_risk',
 'txn_hour',
 'late_night_hours',
 'amount_high',
 'critical_ip_risk',
 'critical_device_trust',
 'new_account',
 'velocity_spike',
 'chargeback_flag',
 'low_kyc',
 'web_high_amount']

In [ ]:
# Split train and test data

X_train = train_df[all_features]
y_train = train_df["is_fraud"]

X_test = test_df[all_features]
y_test = test_df["is_fraud"]

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

X_train shape: (8752, 26), y_train shape: (8752,)
X_test shape:  (2188, 26), y_test shape:  (2188,)


In [ ]:
# Split categorical_features by encoding type needed
ordinal_cols = ["kyc_tier"]
binary_cols = ["new_device", "location_mismatch"]
nominal_cols = ["home_country", "source_currency", "dest_currency", "channel", "ip_country", "txn_day_of_week"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("ord", OrdinalEncoder(categories=[["low", "standard", "enhanced"]]), ordinal_cols),
        ("nom", OneHotEncoder(handle_unknown="ignore", drop="first"), nominal_cols),
        ("bin", FunctionTransformer(lambda x: x.astype(int)), binary_cols),
    ],
    remainder="drop"  # Drop any columns not specified to be preprocessed 
)

# Fit only on training data, then transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape:  {X_test_processed.shape}")

X_train_processed shape: (8752, 42)
X_test_processed shape:  (2188, 42)


Given the clear rise in fraud rate over time observed during EDA (1.7% in 2022 to 13.4% in 2025), a **chronological split** was used instead of a random split, to realistically simulate how the model would perform when deployed — predicting on future transactions using only past data.

The dataset was sorted by timestamp and split **80/20**, with the earliest 80% of transactions used for training and the most recent 20% held out for testing. As expected, the test set shows a higher fraud rate than the training set, reflecting the genuine shift in fraud behaviour over time rather than a data issue.

Features and target were separated into `X_train`/`y_train` and `X_test`/`y_test`, using a finalised feature list built from the multicollinearity-checked numerical features and the categorical features identified during EDA/Feature Selection.

**Preprocessing was handled with a `ColumnTransformer`**, applying different encoding strategies based on each feature's nature:
- **Numerical features** — scaled with `StandardScaler`
- **`kyc_tier`** — `OrdinalEncoder`, with an explicit order (low < standard < enhanced) to preserve its real risk hierarchy
- **Nominal categoricals** (`home_country`, `source_currency`, `dest_currency`, `channel`, `ip_country`, `txn_day_of_week`) — `OneHotEncoder`, since these categories have no natural order
- **Binary columns** (`new_device`, `location_mismatch`) — cast to integer within the pipeline itself.

The preprocessor was **fit only on the training set**, then applied to the test set, ensuring no information from the test period leaked into how the encoders or scaler were learned.